# 04 — Evaluation, Error Analysis & Business Impact

All four models are built. This notebook is where I decide which one to use and, more importantly, whether any of them are actually useful for a real-world application.


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, joblib
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
ROOT = Path("..")

df = pd.read_csv(ROOT / "data" / "processed" / "engineered_dataset.csv")
df["Previous_Lag"] = df["Previous_Price"].shift(1).bfill()
df.dropna(inplace=True)

X = df.drop(columns=["Food_Price"])
y = df["Food_Price"]
split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f"Evaluation dataset: {df.shape}")


## Full Metrics Table — All Models

**Metrics used and why:**

> **RMSE (Root Mean Squared Error)**
> Measures average prediction error in the same unit as the target (USD). Penalises large errors more than small ones due to squaring.
> *Why for this problem:* A \$2 miss on a \$5 item is a big deal for a retailer. RMSE captures that severity appropriately.
> *Vs MAE:* MAE treats all errors equally. RMSE is more sensitive to large misses — which is what we care about in pricing decisions.

> **MAE (Mean Absolute Error)**
> Average absolute error in USD — the most interpretable metric for non-technical stakeholders.
> "The model is off by \$X on average" is a sentence anyone can understand.

> **MAPE (Mean Absolute Percentage Error)**
> Scale-free version of MAE. Useful for comparing across food items with different price ranges.
> *Caveat:* Undefined if any actual price is 0, and biased for low-value predictions. Safe here since min price is \$0.97.

> **R² (Coefficient of Determination)**
> Fraction of price variance the model explains. R²=1 is perfect; R²=0 is "predicting the mean"; R²<0 is worse than predicting the mean.
> *What negative R² means for ARIMA and LSTM:* Those models are actively harmful — a naive "just predict \$5.47 every time" would beat them.


In [ ]:
def metrics(actual, predicted, name):
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae  = mean_absolute_error(actual, predicted)
    mape = mean_absolute_percentage_error(actual, predicted) * 100
    r2   = r2_score(actual, predicted)
    return {"Model": name, "RMSE": round(rmse,4), "MAE": round(mae,4), "MAPE (%)": round(mape,2), "R²": round(r2,4)}

results = []

# ARIMA
ts = df.groupby(["Year","Month"])["Food_Price"].mean().reset_index()
ts["date"] = pd.to_datetime(ts[["Year","Month"]].assign(DAY=1))
ts = ts.sort_values("date").set_index("date")
arima_model = joblib.load(ROOT / "outputs/models/arima_model.pkl")
arima_preds = arima_model.predict(start=0, end=len(ts)-1).values
results.append(metrics(ts["Food_Price"].values, arima_preds, "ARIMA(5,1,0)"))

# LSTM
series = ts["Food_Price"].values.reshape(-1,1)
scaler = joblib.load(ROOT / "outputs/models/lstm_scaler.pkl")
scaled = scaler.transform(series)
window = 12
gen    = TimeseriesGenerator(scaled, scaled, length=window, batch_size=1)
lstm   = load_model(str(ROOT / "outputs/models/lstm_model.h5"), compile=False)
lstm_preds = scaler.inverse_transform(lstm.predict(gen, verbose=0)).flatten()
results.append(metrics(series[window:].flatten(), lstm_preds, "LSTM (window=12)"))

# Linear Regression baseline
categorical = ["Season"]
numerical   = X.drop(columns=categorical).columns.tolist()
prep = ColumnTransformer([("cat", OneHotEncoder(drop="first"), categorical), ("scale", StandardScaler(), numerical)])
lr_pipe = Pipeline([("prep", prep), ("model", LinearRegression())])
lr_pipe.fit(X_train, y_train)
lr_preds = lr_pipe.predict(X_test)
results.append(metrics(y_test.values, lr_preds, "Linear Regression"))

# XGBoost
xgb_model = joblib.load(ROOT / "outputs/models/xgboost_model.pkl")
xgb_preds = xgb_model.predict(X_test)
results.append(metrics(y_test.values, xgb_preds, "XGBoost (test set)"))

summary = pd.DataFrame(results).set_index("Model")
print(summary.to_string())


In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#DD6666","#E8994D","#5DA8D9","#4CAF50"]

axes[0].bar(summary.index, summary["RMSE"], color=colors, edgecolor="black", lw=0.6)
axes[0].set_title("RMSE — XGBoost is 85% better than ARIMA", fontsize=11)
axes[0].set_ylabel("RMSE (USD)")
for i,v in enumerate(summary["RMSE"]):
    axes[0].text(i, v+0.03, f"{v:.4f}", ha="center", fontsize=9.5)
axes[0].tick_params(axis='x', labelsize=9)

axes[1].bar(summary.index, summary["R²"], color=colors, edgecolor="black", lw=0.6)
axes[1].axhline(0, color="black", lw=0.8, ls="--")
axes[1].set_title("R² — ARIMA and LSTM below zero means worse than a flat mean", fontsize=11)
axes[1].set_ylabel("R²")
for i,v in enumerate(summary["R²"]):
    axes[1].text(i, v+(0.02 if v>=0 else -0.08), f"{v:.3f}", ha="center", fontsize=9.5)
axes[1].tick_params(axis='x', labelsize=9)

plt.suptitle("XGBoost vs baselines — it's not close", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(ROOT / "outputs/figures/08_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


## Feature Importance — What the XGBoost Model Actually Learned


In [ ]:
xgb_regressor  = xgb_model.named_steps["regressor"]
xgb_prep       = xgb_model.named_steps["preprocessor"]
feat_names     = xgb_prep.get_feature_names_out()
importances    = xgb_regressor.feature_importances_
pct            = importances / importances.sum() * 100
idx            = np.argsort(pct)[::-1][:15]
clean_names    = [n.replace("scale__","").replace("cat__","") for n in feat_names[idx]]

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(clean_names[::-1], pct[idx][::-1], color=sns.color_palette("Blues_d",15))
ax.set_title("Previous_Price does 53% of the work — everything else fills in the gaps", fontsize=11, pad=12)
ax.set_xlabel("Relative Importance (%)")
for bar, val in zip(bars, pct[idx][::-1]):
    ax.text(val+0.3, bar.get_y()+bar.get_height()/2, f"{val:.1f}%", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(ROOT / "outputs/figures/05_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print("Top 10 feature importances:")
for i in idx[:10]:
    print(f"  {clean_names[idx.tolist().index(i)]:35s} {pct[i]:.2f}%")


**What this tells me:**

`Previous_Price` at 52.72% isn't just important — it's doing most of the heavy lifting. This makes intuitive sense: yesterday's price is the strongest anchor for today's price in most commodity markets. The question is whether the remaining features genuinely add predictive value or whether we're just building a price-lag model with expensive decorations.

The answer: `Temperature_3m_avg` (5%), `Demand_Trend` (4%), and `Rainfall` (4%) together account for another 13% — not nothing. The interaction terms and rolling features are doing real work. Remove `Previous_Price` and the model would still outperform ARIMA.


## Residual Analysis

Where does the model fail? Patterns in residuals reveal what the model doesn't understand.


In [ ]:
all_preds   = xgb_model.predict(X)
residuals   = y.values - all_preds
abs_resid   = np.abs(residuals)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(all_preds, residuals, alpha=0.35, s=16, color="#4C72B0")
axes[0].axhline(0, color="red", lw=1.5, ls="--")
axes[0].set_xlabel("Predicted Price (USD)"); axes[0].set_ylabel("Residual")
axes[0].set_title("Residuals vs fitted — no funnel shape, errors are homoscedastic", fontsize=10)

sns.histplot(residuals, bins=40, kde=True, ax=axes[1], color="#4C72B0")
axes[1].axvline(0, color="red", lw=1.5, ls="--")
axes[1].set_title(f"Residual distribution — tight and near-normal (std=${np.std(residuals):.3f})", fontsize=10)
axes[1].set_xlabel("Residual (USD)")

plt.tight_layout()
plt.savefig(ROOT / "outputs/figures/06_residuals.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Mean residual: {residuals.mean():.6f}  (should be near 0)")
print(f"Std residual:  {residuals.std():.4f}")
print(f"Max over-prediction: {residuals.min():.4f}")
print(f"Max under-prediction: {residuals.max():.4f}")


## Error Analysis — Where Does the Model Struggle Most?

I pulled the 20 worst predictions to look for patterns.


In [ ]:
df_err = df.copy()
df_err["predicted"] = all_preds
df_err["residual"]  = residuals
df_err["abs_error"] = abs_resid

worst = df_err.nlargest(20, "abs_error")[
    ["Year","Month","Day","Season","Rainfall","Temperature","Demand_Index",
     "Previous_Price","Food_Price","predicted","abs_error"]
].reset_index(drop=True)

print("20 worst predictions:")
print(worst.round(3).to_string())


In [ ]:
# Error by Rainfall bracket
df_err["Rainfall_Group"] = pd.cut(
    df_err["Rainfall"], bins=[0,100,200,300],
    labels=["Low (<100mm)","Mid (100–200mm)","High (>200mm)"]
)
err_by_rain = df_err.groupby("Rainfall_Group", observed=True)["abs_error"].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(err_by_rain["Rainfall_Group"], err_by_rain["abs_error"],
            color=["#5DA8D9","#E8994D","#DD6666"], edgecolor="black", lw=0.6)
axes[0].set_title("High-rainfall periods are where the model struggles most", fontsize=10)
axes[0].set_ylabel("Mean Absolute Error (USD)")
for i,v in enumerate(err_by_rain["abs_error"]):
    axes[0].text(i, v+0.003, f"${v:.3f}", ha="center", fontsize=11)

# Error vs Demand_Index
axes[1].scatter(df_err["Demand_Index"], df_err["abs_error"], alpha=0.35, s=14, color="#4C72B0")
axes[1].set_title("Extreme Demand_Index (high and low) produces larger errors", fontsize=10)
axes[1].set_xlabel("Demand Index"); axes[1].set_ylabel("Absolute Error (USD)")

plt.suptitle("Where the model fails: high-rainfall + extreme demand combinations", y=1.02, fontsize=11)
plt.tight_layout()
plt.savefig(ROOT / "outputs/figures/10_error_by_rainfall.png", dpi=150, bbox_inches="tight")
plt.show()


**Error patterns I found:**

Looking at the 20 worst predictions, three things stand out:
1. **High Rainfall (>200mm)** — errors are consistently larger. High rainfall creates sudden crop damage or supply gluts that don't follow historical patterns. The model has seen this before, but the magnitude varies unpredictably.
2. **Extreme Demand_Index values** (near 0.5 or 1.5) — crisis conditions and peak festive demand are underrepresented in 1,000 rows. The model reverts toward average predictions at the extremes.
3. **Short-duration shocks** — single-day price spikes (e.g., a sudden \$0.63 jump on row 68) are impossible to predict from the features we have. They'd require event data (weather alerts, news).

The model is most reliable in "normal" conditions and least reliable during simultaneous supply-demand shocks.


## Business Impact — What Do These Numbers Mean in Practice?

> **RMSE = \$0.21 on a mean price of \$5.47 → that's a 3.8% average error.**

At a retail level:
- A grocery chain planning weekly promotions could use this model to decide whether to discount Tomatoes (model predicts price drop) or hold margins (model predicts price rise).
- At MAPE = 4.27%, for a \$3.00 item, the model is off by \$0.13 on average. That's workable for planning decisions.
- For a \$1.50 item, it's off by \$0.06 — fine for budget forecasting.

**Where it breaks down in production:**
- Requires `Previous_Price` as input — if yesterday's price isn't available (new product, data outage), performance degrades to Linear Regression territory (RMSE ~\$0.35).
- Built on synthetic data. Real-world validation against FAO/USDA price series would be the critical next step before any operational use.
- No exogenous shock inputs — geopolitical events, natural disasters, and policy changes aren't in the model.


## Challenges — What Actually Went Wrong

**Challenge 1: Designing the dataset from scratch**
- *The problem:* No existing dataset had all the variables we wanted at daily resolution. Building synthetic data sounds easy — it isn't. Every feature range had to be justified: why is Transport_Cost between \$50–\$500? (USDA logistics reports.) Why is Exchange_Rate 0.1–1.0? (We normalized it.) Our professor pushed back hard on several choices.
- *What I tried first:* Uniform random sampling for all features. The resulting correlations were completely unrealistic.
- *What worked:* Each feature was sampled from distributions that reflected real-world literature — e.g., crop yield following a truncated normal around 6 tons/ha with seasonal variance.
- *The result:* A dataset where the feature interactions actually make sense agronomically.

**Challenge 2: SMOTE-equivalent oversampling for extreme price events**
- *The problem:* Prices above \$9 and below \$1.50 are underrepresented (fewer than 50 rows each). The model consistently under-predicts in these ranges.
- *What I tried:* Synthetic oversampling of high/low price events. It inflated the extreme-range metrics but hurt middle-range performance — the model started predicting artificial spikes.
- *What worked:* Dropping the oversampling and acknowledging it in error analysis. Honesty over inflated metrics.
- *The result:* Slightly worse performance at extremes, but the model is trustworthy in the 80% of cases where prices are in the \$2–\$9 range.

**Challenge 3: The CV vs test set RMSE gap (1.22 vs 0.21)**
- *The problem:* First time I saw this gap, I thought something was wrong. CV RMSE of 1.22 looks terrible next to test RMSE of 0.21.
- *What I tried:* Shuffling the data to see if the gap collapsed. It did — which confirmed the issue was temporal, not a bug.
- *What worked:* Recognising that early CV folds have very small training sets (160 rows), and the gap closes as training data grows (Fold 5 RMSE = 1.03). The 800/200 temporal split is the representative evaluation.
- *The result:* Both numbers are reported, with explanation. The gap is real but understood.


## Experiment Log

| # | Experiment | Change | Before | After | Decision |
|---|---|---|---|---|---|
| 01 | ARIMA baseline | Univariate price series only | — | RMSE=1.40, R²=-0.35 | Kept as lower bound |
| 02 | LSTM (window=12) | Monthly aggregated series | RMSE=1.40 | RMSE=1.29, R²=-0.09 | Kept for comparison |
| 03 | Linear Regression | All 27 features, StandardScaler | — | RMSE=0.35, R²=0.98 | Kept as strong baseline |
| 04 | XGBoost default | No hyperparameter tuning | RMSE=0.35 | RMSE=0.22, R²=0.99 | Kept |
| 05 | Add interaction features | Temp×Crop, Rainfall×Demand, etc. | RMSE=0.22 | RMSE=0.21 | Kept (marginal gain, strong justification) |
| 06 | SMOTE for extreme prices | Oversample high/low price rows | RMSE=0.21 | Middle-range hurt | Dropped |
| 07 | Random split (test) | Shuffle before split | RMSE=0.21 | RMSE≈0.02 (leaking!) | Dropped immediately |
| 08 | max_depth=6 | Deeper trees | RMSE=0.21 | Train RMSE: 0.08, Test: 0.25 | Dropped (overfit) |


## What I'd Do Next

1. **Validate against real data.** The obvious gap in this project is that the dataset is synthetic. FAO price data for India or the US is publicly available — running this pipeline against actual historical prices would tell us if the feature design holds up.

2. **Multivariate LSTM using all daily features.** The current LSTM sees only the univariate price series. Feeding it all 27 features as a multivariate time series (with proper windowing) could unlock the temporal dynamics it failed to capture here.

3. **Add velocity features.** Rolling 7-day and 30-day price velocity (rate of change, not just level) was on the feature engineering list but didn't make the final cut due to the small dataset size. With more data, these would likely rank in the top 5 features.

4. **Prediction intervals instead of point predictions.** The current model gives a single price forecast. A quantile regression variant of XGBoost (or a bootstrap ensemble) would give confidence bounds — much more useful for retail planning ("price will be between \$4.80 and \$5.20 with 90% probability").

5. **Test distribution shift.** The model was trained on 2010–2023. Applying it to 2024–2025 data (once generated or obtained) would reveal whether the learned patterns are stable or whether the feature relationships drift over time.
